In [ ]:
import numpy as np
import rasterio
from rasterio.enums import Resampling
import richdem as rd

# -------------------------------------------------------
# INPUT 1: SLOPE — derive directly from your 10m DEM
# -------------------------------------------------------
with rasterio.open("./asc_file/dem_post.tif") as src:
    dem_array = src.read(1).astype(float)
    profile = src.profile
    nodata = src.nodata
    transform = src.transform
    crs = src.crs

# Replace nodata with nan
if nodata is not None:
    dem_array[dem_array == nodata] = np.nan

# Derive slope
dem_rd = rd.rdarray(dem_array, no_data=np.nan)
dem_rd.geotransform = transform.to_gdal()
slope = rd.TerrainAttribute(dem_rd, attrib="slope_degrees")

# Save
profile.update(dtype=rasterio.float32, nodata=-9999)
with rasterio.open("slope_10m_post.tif", "w", **profile) as dst:
    dst.write(slope.astype(np.float32), 1)
print("Slope saved.")

# -------------------------------------------------------
# INPUT 2: POINT DENSITY — uniform QL2 constant at 10m
# -------------------------------------------------------
# USGS QL2 = minimum 1 pt/m²
# At 10m pixel: average expected = 1 pt/m² × 100 m² = 100 pts/pixel
# We store as pts/m² to stay consistent with FIS literature (= 1.0)
point_density = np.where(~np.isnan(dem_array), 1.0, np.nan)

with rasterio.open("point_density_10m.tif", "w", **profile) as dst:
    dst.write(point_density.astype(np.float32), 1)
print("Point density saved (uniform QL2 = 1.0 pt/m²).")

Slope saved.



A Slope calculation (degrees)
C Horn, B.K.P., 1981. Hill shading and the reflectance map. Proceedings of the IEEE 69, 14–47. doi:10.1109/PROC.1981.11918

t Wall-time = 0.00830214===================== ] (99% - 0.0s - 1 threads)


# For Vegetation

In [2]:
import pygeohydro
print(dir(pygeohydro))

['EHydro', 'NFHL', 'NID', 'NLD', 'NWIS', 'PackageNotFoundError', 'STNFloodEventData', 'SensorThings', 'WBD', 'WaterQuality', '__all__', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'annotations', 'cover_legends', 'cover_statistics', 'descriptor_legends', 'exceptions', 'get_camels', 'get_us_states', 'helpers', 'huc_wb_full', 'interactive_map', 'irrigation_withdrawals', 'levee', 'nfhl', 'nid', 'nlcd', 'nlcd_area_percent', 'nlcd_bycoords', 'nlcd_bygeom', 'nwis', 'overland_roughness', 'plot', 'print_versions', 'pygeohydro', 'show_versions', 'soil_gnatsgo', 'soil_polaris', 'soil_properties', 'soil_soilgrids', 'ssebopeta_bycoords', 'ssebopeta_bygeom', 'stn_flood_event', 'stnfloodevents', 'streamflow_fillna', 'us_abbrs', 'version', 'waterdata', 'watershed']


In [6]:
import pygeohydro as gh
import numpy as np
import rasterio
from rasterio.warp import reproject, Resampling as RResampling
from shapely.geometry import box
import geopandas as gpd

bbox = (-119.690710, 34.423614, -119.615218, 34.500526)
geom = gpd.GeoDataFrame(geometry=[box(*bbox)], crs="EPSG:4326")

canopy = gh.nlcd_bygeom(geom, resolution=30, years={"canopy": [2019]})

# Key is integer 0
canopy_ds = canopy[0]
print("Dataset:", canopy_ds)
print("Variables:", list(canopy_ds.data_vars))

# Grab the canopy variable — check what it's called
canopy_var = list(canopy_ds.data_vars)[0]
print("Using variable:", canopy_var)

canopy_ds[canopy_var].rio.to_raster("canopy_30m.tif")
print("Canopy 30m saved.")

# -------------------------------------------------------
# Resample to 10m to match your working DEMs
# -------------------------------------------------------
with rasterio.open("canopy_30m.tif") as src:
    canopy_src = src.read(1).astype(np.float32)
    src_transform = src.transform
    src_crs = src.crs

with rasterio.open("./asc_file/dem_pre.tif") as ref:
    ref_profile = ref.profile
    ref_transform = ref.transform
    ref_crs = ref.crs
    ref_shape = ref.shape

canopy_10m = np.empty(ref_shape, dtype=np.float32)

reproject(
    source=canopy_src,
    destination=canopy_10m,
    src_transform=src_transform,
    src_crs=src_crs,
    dst_transform=ref_transform,
    dst_crs=ref_crs,
    resampling=RResampling.bilinear
)

ref_profile.update(dtype=rasterio.float32, nodata=-9999)
with rasterio.open("canopy_10m.tif", "w", **ref_profile) as dst:
    dst.write(canopy_10m, 1)
print("Canopy resampled to 10m saved.")

Dataset: <xarray.Dataset> Size: 269kB
Dimensions:      (x: 232, y: 285)
Coordinates:
  * x            (x) float64 2kB -119.7 -119.7 -119.7 ... -119.6 -119.6 -119.6
  * y            (y) float64 2kB 34.5 34.5 34.5 34.5 ... 34.42 34.42 34.42 34.42
    spatial_ref  int64 8B 0
Data variables:
    canopy_2019  (y, x) float32 264kB 60.0 84.0 81.0 50.0 ... 25.0 54.0 54.0
Attributes:
    TIFFTAG_XRESOLUTION:     1
    TIFFTAG_YRESOLUTION:     1
    TIFFTAG_RESOLUTIONUNIT:  1 (unitless)
    AREA_OR_POINT:           Area
    scale_factor:            1.0
    add_offset:              0.0
    _FillValue:              255
    nodatavals:              (255,)
Variables: ['canopy_2019']
Using variable: canopy_2019
Canopy 30m saved.
Canopy resampled to 10m saved.
